In [4]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

from darts import TimeSeries, concatenate
from darts.models import TFTModel
from darts.dataprocessing.transformers import Scaler
from darts.metrics import mae, mse, mape
from darts.utils.model_selection import train_test_split
from darts.utils.timeseries_generation import datetime_attribute_timeseries as dt_attr
from darts.utils.missing_values import fill_missing_values

# =========================
# Параметры модели
# =========================
INPUT = 60
HORIZON = 8

# =========================
# 1. Загрузка данных
# =========================
data_path = Path('data/sales.csv') if Path('data/sales.csv').exists() else Path('../data/sales.csv')
data = pd.read_csv(
    data_path,
    sep=';',
    parse_dates=['Week']
)

print("=== RAW DATA ===")
print(f"Shape: {data.shape}")
print(f"Columns: {data.columns.tolist()}")
print(data.head())

# Сортируем
data = data.sort_values(['City', 'Week'])

# =========================
# 2. Создание рядов
# =========================
target_series = TimeSeries.from_group_dataframe(
    df=data,
    time_col='Week',
    group_cols='City',
    value_cols='revenue',
    metadata_cols='City',   # можно убрать; модели это не помогает
    fill_missing_dates=True,
    freq='W-MON',
    verbose=False
)

print(f"Создано рядов: {len(target_series)}")

target_series = [
    fill_missing_values(ts, fill=0.0)
    for ts in target_series
]

# Проверка длины рядов
min_len = min(len(ts) for ts in target_series)
print("Минимальная длина ряда:", min_len)

# =========================
# 3. Масштабирование
# =========================
scaler = Scaler()
target_series_scaled = scaler.fit_transform(target_series)

# =========================
# 4. Future covariates
# =========================
future_covariates = []

for ts in target_series_scaled:
    full_range = pd.date_range(
        start=ts.start_time(),
        end=ts.end_time() + pd.Timedelta(weeks=HORIZON),
        freq='W-MON'
    )

    cov = concatenate([
        dt_attr(full_range, "month", one_hot=True),
        dt_attr(full_range, "year"),
    ], axis=1)
    future_covariates.append(cov)

# =========================
# 5. Train / validation split
# =========================
train, val = train_test_split(
    target_series_scaled,
    axis=1,
    test_size=0.2,
    input_size=INPUT,
    horizon=HORIZON,
    vertical_split_type="model-aware"
)

train_cov = []
val_cov = []

for cov, ts_train, ts_val in zip(future_covariates, train, val):
    train_cov.append(cov.slice_intersect(ts_train))
    val_cov.append(cov.slice_intersect(ts_val))

# =========================
# 6. Модель
# =========================
model = TFTModel(
    input_chunk_length=INPUT,
    output_chunk_length=HORIZON,
    hidden_size=32,
    lstm_layers=1,
    num_attention_heads=4,
    dropout=0.1,
    batch_size=64,
    n_epochs=10,
    add_relative_index=True,
    use_static_covariates=False,
    random_state=42
)

model.fit(
    series=train,
    future_covariates=train_cov,
    verbose=True
)

=== RAW DATA ===
Shape: (10464, 3)
Columns: ['Week', 'City', 'revenue']
        Week         City     revenue
0 2022-01-03  Arkhangelsk  1237343.94
1 2022-01-10  Arkhangelsk  1139750.43
2 2022-01-17  Arkhangelsk  1131688.16
3 2022-01-24  Arkhangelsk  1135912.90
4 2022-01-31  Arkhangelsk  1327978.58
Создано рядов: 50
Минимальная длина ряда: 94


GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
You are using a CUDA device ('NVIDIA GeForce RTX 3060') that has Tensor Cores. To properly utilize them, you should set `torch.set_float32_matmul_precision('medium' | 'high')` which will trade-off precision for performance. For more details, read https://pytorch.org/docs/stable/generated/torch.set_float32_matmul_precision.html#torch.set_float32_matmul_precision
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]

   | Name                              | Type                             | Params | Mode  | FLOPs
--------------------------------------------------------------------------------------------------------
0  | train_metrics                     | MetricCollection                 | 0  

Epoch 0:   0%|          | 0/83 [00:00<?, ?it/s]

Exception ignored in: <function tqdm.__del__ at 0x000001C8A6C032E0>
Traceback (most recent call last):
  File "d:\VS_project\tft_monthly_remainder_forecast\.venv\Lib\site-packages\tqdm\std.py", line 1148, in __del__
    self.close()
  File "d:\VS_project\tft_monthly_remainder_forecast\.venv\Lib\site-packages\tqdm\notebook.py", line 277, in close
    self.disp(bar_style='danger', check_delay=False)
    ^^^^^^^^^
AttributeError: 'tqdm_notebook' object has no attribute 'disp'


Epoch 9: 100%|██████████| 83/83 [00:10<00:00,  7.97it/s, train_loss=0.239]

`Trainer.fit` stopped: `max_epochs=10` reached.


Epoch 9: 100%|██████████| 83/83 [00:10<00:00,  7.97it/s, train_loss=0.239]


TFTModel(output_chunk_shift=0, hidden_size=32, lstm_layers=1, num_attention_heads=4, full_attention=False, feed_forward=GatedResidualNetwork, dropout=0.1, hidden_continuous_size=8, categorical_embedding_sizes=None, add_relative_index=True, skip_interpolation=False, loss_fn=None, likelihood=None, norm_type=LayerNorm, use_static_covariates=False, input_chunk_length=60, output_chunk_length=8, batch_size=64, n_epochs=10, random_state=42)